In [2]:
# Iris dataset
from sklearn.datasets import load_iris

In [35]:
data = load_iris()
X = data.data
y = data.target

In [7]:
import numpy as np

In [9]:
p_0 = np.sum(y == 0) / len(y)
p_1 = np.sum(y == 1) / len(y)
p_2 = np.sum(y == 2) / len(y)
p_0, p_1, p_2

(0.3333333333333333, 0.3333333333333333, 0.3333333333333333)

In [23]:
# Para la clase 0
mu_0 = np.mean(X[y==0,:], axis = 0)
sigma_0 = np.std(X[y==0,:], axis = 0)

print(f"Clase 0:\nmu: {mu_0}\nsigma: {sigma_0}")

# Para la clase 1
mu_1 = np.mean(X[y==1,:], axis = 0)
sigma_1 = np.std(X[y==1,:], axis = 0)
print(f"Clase 1:\nmu: {mu_1}\nsigma: {sigma_1}")

# Para la clase 2
mu_2 = np.mean(X[y==2,:], axis = 0)
sigma_2 = np.std(X[y==2,:], axis = 0)
print(f"Clase 1:\nmu: {mu_2}\nsigma: {sigma_2}")

# Matriz de medias
mu = np.array([mu_0, mu_1, mu_2])

# Matriz de varianzas
sigma = np.array([sigma_0, sigma_1, sigma_2])

Clase 0:
mu: [5.006 3.428 1.462 0.246]
sigma: [0.34894699 0.37525458 0.17191859 0.10432641]
Clase 1:
mu: [5.936 2.77  4.26  1.326]
sigma: [0.51098337 0.31064449 0.46518813 0.19576517]
Clase 1:
mu: [6.588 2.974 5.552 2.026]
sigma: [0.62948868 0.31925538 0.54634787 0.27188968]


In [21]:
evidencia = [5.5,3.0,4.5,1.5]


In [22]:
def get_proba(evidencia, mu, sigma):
    prob = 1
    for i in range(len(mu)):
        prob *= (1 / (sigma[i] * np.sqrt(2 * np.pi))) * np.exp(-((evidencia[i] - mu[i]) ** 2) / (2 * (sigma[i] ** 2)))
    return prob

In [24]:
def get_proba_class(evidencia, mu, sigma, p):
    prob = get_proba(evidencia, mu, sigma)
    return prob * p

In [27]:
proba_0 = get_proba_class(evidencia, mu_0, sigma_0, p_0)
proba_1 = get_proba_class(evidencia, mu_1, sigma_1, p_1)
proba_2 = get_proba_class(evidencia, mu_2, sigma_2, p_2)
print(f"Probabilidad de la clase 0: {proba_0}")
print(f"Probabilidad de la clase 1: {proba_1}")
print(f"Probabilidad de la clase 2: {proba_2}")

Probabilidad de la clase 0: 4.530834331543873e-100
Probabilidad de la clase 1: 0.18197364721821518
Probabilidad de la clase 2: 0.0015261173307663004


In [29]:
delta = proba_0 + proba_1 + proba_2
proba_0 = proba_0 /delta
proba_1 = proba_1 /delta
proba_2 = proba_2 /delta
print(f"Probabilidad de la clase 0: {proba_0}")
print(f"Probabilidad de la clase 1: {proba_1}")
print(f"Probabilidad de la clase 2: {proba_2}")

Probabilidad de la clase 0: 2.469122694887415e-99
Probabilidad de la clase 1: 0.9916832736297112
Probabilidad de la clase 2: 0.008316726370288801


In [30]:
np.argmax([proba_0, proba_1, proba_2])

1

In [49]:
class NaiveBayes:
    def __init__(self, categorical_variables = None):
        self.categorical_variables = categorical_variables
        
    
    def fit(self, X, y):
        self.X = X
        self.y = y
        n_classes = len(np.unique(y))
        classes = np.unique(y)
        self.mu = np.zeros((n_classes, X.shape[1]))
        self.sigma = np.zeros((n_classes, X.shape[1]))
        self.p_class = np.zeros(n_classes)
        
        for index, class_ in enumerate(classes):
            self.mu[index] = np.mean(X[y==class_], axis=0)
            self.sigma[index] = np.std(X[y==class_], axis=0)
            self.p_class[index] = np.sum(y == class_) / len(y)
    
        return self
    
    def predict(self, X):
        y_pred = np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            probs = np.zeros(len(self.p_class))
            for j in range(len(self.p_class)):
                probs[j] = self._get_proba_class(X[i], self.mu[j], self.sigma[j], self.p_class[j])
            y_pred[i] = np.argmax(probs)
        return y_pred
    
    def _get_proba(self, evidencia, mu, sigma):
        prob = 1
        for i in range(len(mu)):
            prob *= (1 / (sigma[i] * np.sqrt(2 * np.pi))) * np.exp(-((evidencia[i] - mu[i]) ** 2) / (2 * (sigma[i] ** 2)))
        return prob
    
    def _get_proba_class(self, evidencia, mu, sigma, p):
        prob = self._get_proba(evidencia, mu, sigma)
        return prob * p

In [50]:
nb = NaiveBayes()

nb.fit(X,y)

In [51]:
y_pred = nb.predict(X)

from sklearn.metrics import accuracy_score

accuracy_score(y, y_pred)

0.96